<a href="https://colab.research.google.com/github/arturfurlan07-debug/MINI_PROJETO-_3/blob/main/MINI_PROJETO_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q fastapi uvicorn python-dotenv requests

import os
for p in ["meu_projeto_api/server/app", "meu_projeto_api/client"]:
    os.makedirs(p, exist_ok=True)
open("meu_projeto_api/server/app/__init__.py", "a").close()

In [ ]:
from pathlib import Path
raiz = Path("meu_projeto_api")

(raiz / "requirements.txt").write_text("fastapi\nuvicorn\npython-dotenv\nrequests\n")
(raiz / ".gitignore").write_text(".env\n__pycache__/\n*.log\n*.zip\n")
(raiz / "server" / ".env.example").write_text("API_KEY=chave-secreta-123\n")
(raiz / "client" / ".env.example").write_text(
    "API_URL=http://127.0.0.1:8000\nAPI_KEY=chave-secreta-123\n"
)

# Em projeto real o .env não vai pro GitHub; aqui criamos a partir do exemplo
(raiz / "server" / ".env").write_text((raiz / "server" / ".env.example").read_text())
(raiz / "client" / ".env").write_text((raiz / "client" / ".env.example").read_text())

56

In [ ]:
%%writefile meu_projeto_api/server/app/main.py
"""Servidor da API de Análise de Sentimento (FastAPI)."""
import os
import re
import unicodedata
from collections import Counter, deque
from datetime import datetime
from typing import List

from dotenv import load_dotenv
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

load_dotenv()
API_KEY = os.getenv("API_KEY", "")

app = FastAPI(
    title="API de Análise de Sentimento",
    description="Classifica textos em português como positivo, negativo ou neutro.",
    version="1.0.0",
)

historico = deque(maxlen=100)

POSITIVAS = {
    "bom", "boa", "otimo", "otima", "excelente", "maravilhoso", "maravilhosa",
    "gostei", "adorei", "amei", "feliz", "legal", "incrivel", "perfeito",
    "perfeita", "facil", "rapido", "rapida", "recomendo", "top", "satisfeito",
    "satisfeita", "melhor", "lindo", "linda", "show", "divertido", "confiavel",
    "eficiente", "util", "obrigado", "obrigada", "ajudou", "genial", "sensacional",
}
NEGATIVAS = {
    "ruim", "pessimo", "pessima", "horrivel", "terrivel", "odiei", "detestei",
    "triste", "lento", "lenta", "dificil", "caro", "cara", "problema", "erro",
    "bug", "decepcionado", "decepcionada", "decepcionante", "insatisfeito",
    "insatisfeita", "fraco", "fraca", "pior", "chato", "chata", "falha",
    "travou", "demorado", "demorada", "inutil", "complicado", "complicada",
}
NEGACOES = {"nao", "nunca", "jamais", "nem", "sem"}
INTENSIFICADORES = {"muito", "muita", "super", "extremamente", "bastante", "demais"}


class TextoEntrada(BaseModel):
    texto: str = Field(..., min_length=1, max_length=2000)


class LoteEntrada(BaseModel):
    textos: List[str] = Field(..., min_length=1, max_length=50)


def verificar_chave(x_api_key: str = Header(default="")):
    """Se API_KEY estiver definida no .env, exige o header X-API-Key."""
    if API_KEY and x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Chave de API inválida")


def tokenizar(texto: str) -> List[str]:
    texto = unicodedata.normalize("NFD", texto.lower())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return re.findall(r"[a-z]+", texto)


def analisar_texto(texto: str) -> dict:
    tokens = tokenizar(texto)
    pontuacao = 0.0
    positivas, negativas = [], []

    for i, tok in enumerate(tokens):
        if tok in POSITIVAS:
            peso = 1.0
        elif tok in NEGATIVAS:
            peso = -1.0
        else:
            continue

        contexto = tokens[max(0, i - 2):i]
        if any(c in INTENSIFICADORES for c in contexto):
            peso *= 1.5
        if any(c in NEGACOES for c in contexto):
            peso *= -1

        pontuacao += peso
        (positivas if peso > 0 else negativas).append(tok)

    if pontuacao > 0.5:
        sentimento = "positivo"
    elif pontuacao < -0.5:
        sentimento = "negativo"
    else:
        sentimento = "neutro"

    return {
        "texto": texto,
        "sentimento": sentimento,
        "pontuacao": round(pontuacao, 2),
        "palavras_positivas": positivas,
        "palavras_negativas": negativas,
    }


def registrar(resultado: dict) -> dict:
    historico.appendleft({**resultado, "data": datetime.now().isoformat(timespec="seconds")})
    return resultado


@app.get("/")
def raiz():
    return {"mensagem": "API de Análise de Sentimento no ar", "docs": "/docs"}


@app.get("/saude")
def saude():
    return {"status": "ok"}


@app.post("/analisar", dependencies=[Depends(verificar_chave)])
def analisar(entrada: TextoEntrada):
    return registrar(analisar_texto(entrada.texto))


@app.post("/analisar/lote", dependencies=[Depends(verificar_chave)])
def analisar_lote(entrada: LoteEntrada):
    return {"resultados": [registrar(analisar_texto(t)) for t in entrada.textos]}


@app.get("/historico", dependencies=[Depends(verificar_chave)])
def ver_historico(limite: int = 10):
    return {"total": len(historico), "itens": list(historico)[:limite]}


@app.get("/estatisticas", dependencies=[Depends(verificar_chave)])
def estatisticas():
    contagem = Counter(item["sentimento"] for item in historico)
    return {
        "total": len(historico),
        "positivo": contagem["positivo"],
        "negativo": contagem["negativo"],
        "neutro": contagem["neutro"],
    }

Overwriting meu_projeto_api/server/app/main.py


In [ ]:
%%writefile meu_projeto_api/client/main.py
"""Cliente que consome a API de Análise de Sentimento."""
import argparse
import os
import sys

import requests
from dotenv import load_dotenv

load_dotenv()
API_URL = os.getenv("API_URL", "http://127.0.0.1:8000").rstrip("/")
API_KEY = os.getenv("API_KEY", "")
HEADERS = {"X-API-Key": API_KEY}
ICONES = {"positivo": "😀", "negativo": "😞", "neutro": "😐"}


def chamar(metodo, caminho, **kwargs):
    try:
        resp = requests.request(
            metodo, f"{API_URL}{caminho}", headers=HEADERS, timeout=10, **kwargs
        )
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        sys.exit("Não foi possível conectar à API. O servidor está rodando?")
    except requests.exceptions.HTTPError:
        sys.exit(f"Erro HTTP {resp.status_code}: {resp.text}")


def mostrar(r):
    print(f"{ICONES[r['sentimento']]} {r['sentimento'].upper()} "
          f"(pontuação {r['pontuacao']}) -> {r['texto']}")


def analisar(texto):
    return chamar("POST", "/analisar", json={"texto": texto})


def analisar_lote(textos):
    return chamar("POST", "/analisar/lote", json={"textos": textos})["resultados"]


def demonstracao():
    print("== Saúde da API ==")
    print(chamar("GET", "/saude"))

    print("\n== Análise individual ==")
    mostrar(analisar("Adorei o atendimento, foi muito bom!"))

    print("\n== Análise em lote ==")
    for r in analisar_lote([
        "O aplicativo é ótimo e muito rápido",
        "Achei péssimo, travou o tempo todo",
        "Não gostei do produto",
        "Recebi o pedido hoje",
    ]):
        mostrar(r)

    print("\n== Estatísticas ==")
    print(chamar("GET", "/estatisticas"))

    print("\n== Últimas análises ==")
    for item in chamar("GET", "/historico", params={"limite": 3})["itens"]:
        print(f"[{item['data']}] ", end="")
        mostrar(item)


def main():
    parser = argparse.ArgumentParser(description="Cliente da API de sentimento")
    parser.add_argument("textos", nargs="*", help="textos a analisar (vazio = demonstração)")
    args = parser.parse_args()

    if not args.textos:
        demonstracao()
    elif len(args.textos) == 1:
        mostrar(analisar(args.textos[0]))
    else:
        for r in analisar_lote(args.textos):
            mostrar(r)


if __name__ == "__main__":
    main()

Overwriting meu_projeto_api/client/main.py


In [ ]:
import subprocess, sys, time, requests

servidor = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--port", "8000"],
    cwd="meu_projeto_api/server",
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)

for _ in range(20):
    try:
        print(requests.get("http://127.0.0.1:8000/saude").json())
        break
    except requests.exceptions.ConnectionError:
        time.sleep(0.5)
else:
    print(open("server.log").read())

{'status': 'ok'}


In [ ]:
!python meu_projeto_api/client/main.py
!python meu_projeto_api/client/main.py "Esse curso é excelente" "Não gostei da prova"

== Saúde da API ==
{'status': 'ok'}

== Análise individual ==
😀 POSITIVO (pontuação 2.5) -> Adorei o atendimento, foi muito bom!

== Análise em lote ==
😀 POSITIVO (pontuação 2.5) -> O aplicativo é ótimo e muito rápido
😞 NEGATIVO (pontuação -2.0) -> Achei péssimo, travou o tempo todo
😞 NEGATIVO (pontuação -1.0) -> Não gostei do produto
😐 NEUTRO (pontuação 0.0) -> Recebi o pedido hoje

== Estatísticas ==
{'total': 12, 'positivo': 5, 'negativo': 5, 'neutro': 2}

== Últimas análises ==
[2026-11-11T17:29:45] 😐 NEUTRO (pontuação 0.0) -> Recebi o pedido hoje
[2026-11-11T17:29:45] 😞 NEGATIVO (pontuação -1.0) -> Não gostei do produto
[2026-11-11T17:29:45] 😞 NEGATIVO (pontuação -2.0) -> Achei péssimo, travou o tempo todo
😀 POSITIVO (pontuação 1.0) -> Esse curso é excelente
😞 NEGATIVO (pontuação -1.0) -> Não gostei da prova
